# California Housing and MNIST: Adaptive k-NN and Decision-Tree Wrappers

## Project Overview

This notebook implements reusable k-nearest-neighbor and decision-tree wrappers that automatically support both regression and classification.

Each wrapper provides:

- automatic task detection from the target variable
- default regression/classification models
- optional validation-based hyperparameter optimization
- preprocessing selection for k-NN
- final retraining on the complete training set
- a shared `fit` / `predict` interface

The implementations are evaluated on California Housing for regression and MNIST for classification.

### Technical Coverage

- shared `fit` / `predict` interface
- automatic regression/classification detection
- k-NN classification and regression
- decision-tree classification and regression
- StandardScaler and MinMaxScaler selection
- validation-based hyperparameter search
- final retraining with selected parameters
- California Housing regression
- MNIST classification


## 1. Environment

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.utils.multiclass import type_of_target
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import time

from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler


## 2. Adaptive k-NN Wrapper

The wrapper automatically detects whether the target represents a classification or regression problem.

In default mode it uses five uniformly weighted neighbors. With `optimize=True`, it compares:

- no preprocessing, `StandardScaler`, and `MinMaxScaler`
- `k = 3, 5, 7`
- uniform and distance weighting

The best configuration is selected using validation accuracy for classification or validation MSE for regression, then retrained on the complete training set.


In [2]:
class knn:

    def __init__(self):
        self.scaler = None

    def _detect_task(self, y):
        target_type = type_of_target(y)

        if target_type in ("binary", "multiclass"):
            return "classification"

        return "regression"

    def _make_model(self):
        if self.task == "classification":
            return KNeighborsClassifier(
                n_neighbors=self.n_neighbors,
                weights=self.weights,
                algorithm="brute",
                n_jobs=-1,
            )

        return KNeighborsRegressor(
            n_neighbors=self.n_neighbors,
            weights=self.weights,
            algorithm="brute",
            n_jobs=-1,
        )

    def _apply_preprocessing(self, X_train, X_other, preprocess):
        if preprocess == "StandardScaler":
            scaler = StandardScaler()
        elif preprocess == "MinMaxScaler":
            scaler = MinMaxScaler()
        else:
            return X_train, X_other, None

        X_train_scaled = scaler.fit_transform(X_train)
        X_other_scaled = scaler.transform(X_other)

        return X_train_scaled, X_other_scaled, scaler

    def fit(self, X_train, y_train, optimize=False):
        self.task = self._detect_task(y_train)

        # Default configuration
        if not optimize:
            self.preprocess = "None"
            self.n_neighbors = 5
            self.weights = "uniform"
            self.scaler = None

            self.model = self._make_model()
            self.model.fit(X_train, y_train)
            return self

        # Validation split for model selection
        stratify = y_train if self.task == "classification" else None

        X_fit, X_val, y_fit, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.20,
            random_state=5361,
            stratify=stratify,
        )

        preprocess_options = ["None", "StandardScaler", "MinMaxScaler"]
        neighbor_options = [3, 5, 7]
        weight_options = ["uniform", "distance"]

        best_score = -np.inf if self.task == "classification" else np.inf
        best_params = None

        for preprocess in preprocess_options:
            X_fit_p, X_val_p, _ = self._apply_preprocessing(
                X_fit, X_val, preprocess
            )

            for n_neighbors in neighbor_options:
                for weights in weight_options:
                    if self.task == "classification":
                        candidate = KNeighborsClassifier(
                            n_neighbors=n_neighbors,
                            weights=weights,
                            algorithm="brute",
                            n_jobs=-1,
                        )
                    else:
                        candidate = KNeighborsRegressor(
                            n_neighbors=n_neighbors,
                            weights=weights,
                            algorithm="brute",
                            n_jobs=-1,
                        )

                    candidate.fit(X_fit_p, y_fit)
                    pred = candidate.predict(X_val_p)

                    if self.task == "classification":
                        score = accuracy_score(y_val, pred)
                        better = score > best_score
                    else:
                        score = mean_squared_error(y_val, pred)
                        better = score < best_score

                    if better:
                        best_score = score
                        best_params = {
                            "preprocess": preprocess,
                            "n_neighbors": n_neighbors,
                            "weights": weights,
                        }

        self.preprocess = best_params["preprocess"]
        self.n_neighbors = best_params["n_neighbors"]
        self.weights = best_params["weights"]

        # Fit the selected preprocessing method on all training data.
        if self.preprocess == "StandardScaler":
            self.scaler = StandardScaler()
            X_train_final = self.scaler.fit_transform(X_train)
        elif self.preprocess == "MinMaxScaler":
            self.scaler = MinMaxScaler()
            X_train_final = self.scaler.fit_transform(X_train)
        else:
            self.scaler = None
            X_train_final = X_train

        # Retrain the selected model using all training observations.
        self.model = self._make_model()
        self.model.fit(X_train_final, y_train)

        return self

    def predict(self, X_test):
        if self.scaler is not None:
            X_test = self.scaler.transform(X_test)

        return self.model.predict(X_test)


## 3. Adaptive Decision-Tree Wrapper

The decision-tree wrapper also detects the task automatically.

In default mode it uses:

- `gini` for classification
- `squared_error` for regression
- unrestricted depth

With `optimize=True`, classification compares Gini and entropy, while regression compares squared error and Friedman MSE across several maximum-depth values. The selected configuration is retrained on all training observations.


In [3]:
class decision_tree:

    def __init__(self):
        return

    def _detect_task(self, y):
        target_type = type_of_target(y)

        if target_type in ("binary", "multiclass"):
            return "classification"

        return "regression"

    def _make_model(self):
        if self.task == "classification":
            return DecisionTreeClassifier(
                criterion=self.criterion,
                max_depth=self.max_depth,
                random_state=5361,
            )

        return DecisionTreeRegressor(
            criterion=self.criterion,
            max_depth=self.max_depth,
            random_state=5361,
        )

    def fit(self, X_train, y_train, optimize=False):
        self.task = self._detect_task(y_train)

        # Default configuration
        if not optimize:
            if self.task == "classification":
                self.criterion = "gini"
            else:
                self.criterion = "squared_error"

            self.max_depth = None
            self.model = self._make_model()
            self.model.fit(X_train, y_train)
            return self

        # Validation split for model selection
        stratify = y_train if self.task == "classification" else None

        X_fit, X_val, y_fit, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.20,
            random_state=5361,
            stratify=stratify,
        )

        if self.task == "classification":
            criterion_options = ["gini", "entropy"]
        else:
            criterion_options = ["squared_error", "friedman_mse"]

        depth_options = [None, 5, 10, 15, 20]

        best_score = -np.inf if self.task == "classification" else np.inf
        best_params = None

        for criterion in criterion_options:
            for max_depth in depth_options:
                if self.task == "classification":
                    candidate = DecisionTreeClassifier(
                        criterion=criterion,
                        max_depth=max_depth,
                        random_state=5361,
                    )
                else:
                    candidate = DecisionTreeRegressor(
                        criterion=criterion,
                        max_depth=max_depth,
                        random_state=5361,
                    )

                candidate.fit(X_fit, y_fit)
                pred = candidate.predict(X_val)

                if self.task == "classification":
                    score = accuracy_score(y_val, pred)
                    better = score > best_score
                else:
                    score = mean_squared_error(y_val, pred)
                    better = score < best_score

                if better:
                    best_score = score
                    best_params = {
                        "criterion": criterion,
                        "max_depth": max_depth,
                    }

        self.criterion = best_params["criterion"]
        self.max_depth = best_params["max_depth"]

        # Retrain the selected model using all training observations.
        self.model = self._make_model()
        self.model.fit(X_train, y_train)

        return self

    def predict(self, X_test):
        return self.model.predict(X_test)


## 4. California Housing Regression

California Housing provides an eight-feature continuous-target problem for evaluating the default regression behavior of both wrappers.

In [4]:
from sklearn.datasets import fetch_california_housing

dataset = fetch_california_housing()
x = dataset.data
y = dataset.target
X_train_ch, X_test_ch, y_train_ch, y_test_ch = train_test_split(x, y, test_size=0.20, random_state=5361)
print(f'Array shapes: {X_train_ch.shape = }, {y_train_ch.shape = }, {X_test_ch.shape = }, {y_test_ch.shape = }')

Array shapes: X_train_ch.shape = (16512, 8), y_train_ch.shape = (16512,), X_test_ch.shape = (4128, 8), y_test_ch.shape = (4128,)


### Default Models

In [5]:
print('\nK-nn model')
model = knn()
model.fit(X_train_ch,y_train_ch)
print(f'Model parameters: {model.task = }, {model.n_neighbors = }, {model.weights = }, {model.preprocess = }')
pred = model.predict(X_test_ch)
print(f'mean_absolute_error: {mean_absolute_error(y_test_ch,pred):6.4f}')
print(f'mean_squared_error: {mean_squared_error(y_test_ch,pred):6.4f}')

print('\nDecision tree model')
model = decision_tree()
model.fit(X_train_ch,y_train_ch)
print(f'Model parameters: {model.task = }, {model.criterion = }, {model.max_depth = }')
pred = model.predict(X_test_ch)
print(f'mean_absolute_error: {mean_absolute_error(y_test_ch,pred):6.4f}')
print(f'mean_squared_error: {mean_squared_error(y_test_ch,pred):6.4f}')


K-nn model
Model parameters: model.task = 'regression', model.n_neighbors = 5, model.weights = 'uniform', model.preprocess = 'None'
mean_absolute_error: 0.8152
mean_squared_error: 1.1420

Decision tree model
Model parameters: model.task = 'regression', model.criterion = 'squared_error', model.max_depth = None
mean_absolute_error: 0.4615
mean_squared_error: 0.5340


### Optimized Models

The following models use validation-based parameter selection and then retrain the selected configuration on the complete California Housing training set.


In [6]:
print('\nK-nn model')
model = knn()
model.fit(X_train_ch,y_train_ch,optimize=True)
print(f'Model parameters: {model.task = }, {model.n_neighbors = }, {model.weights = }, {model.preprocess = }')
pred = model.predict(X_test_ch)
print(f'mean_absolute_error: {mean_absolute_error(y_test_ch,pred):6.4f}')
print(f'mean_squared_error: {mean_squared_error(y_test_ch,pred):6.4f}')

print('\nDecision tree model')
model = decision_tree()
model.fit(X_train_ch,y_train_ch,optimize=True)
print(f'Model parameters: {model.task = }, {model.criterion = }, {model.max_depth = }')
pred = model.predict(X_test_ch)
print(f'mean_absolute_error: {mean_absolute_error(y_test_ch,pred):6.4f}')
print(f'mean_squared_error: {mean_squared_error(y_test_ch,pred):6.4f}')


K-nn model
Model parameters: model.task = 'regression', model.n_neighbors = 7, model.weights = 'distance', model.preprocess = 'MinMaxScaler'
mean_absolute_error: 0.4135
mean_squared_error: 0.3830

Decision tree model
Model parameters: model.task = 'regression', model.criterion = 'squared_error', model.max_depth = 10
mean_absolute_error: 0.4282
mean_squared_error: 0.4235


## 5. MNIST Classification Interface

MNIST is normalized and flattened into 784-dimensional feature vectors to demonstrate the classification-side interface expected by the same wrappers.

In [7]:
# Download the MNIST dataset
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train = np.float32(X_train.reshape(X_train.shape[0],-1)/255)
X_test = np.float32(X_test.reshape(X_test.shape[0],-1)/255)

print(f'Array shapes: {X_train.shape = }, {y_train.shape = }, {X_test.shape = }, {y_test.shape = }')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Array shapes: X_train.shape = (60000, 784), y_train.shape = (60000,), X_test.shape = (10000, 784), y_test.shape = (10000,)


### Default Classification Models

The wrappers now automatically recognize the integer MNIST target as a multiclass classification problem, so the k-NN and decision-tree classifiers are selected automatically.


In [8]:
print('\nK-nn model')
model = knn()
model.fit(X_train,y_train)
print(f'Model parameters: {model.task = }, {model.n_neighbors = }, {model.weights = }, {model.preprocess = }')
pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test,pred):6.4f}')

print('\nDecision tree model')
model = decision_tree()
model.fit(X_train,y_train)
print(f'Model parameters: {model.task = }, {model.criterion = }, {model.max_depth = }')
pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test,pred):6.4f}')


K-nn model
Model parameters: model.task = 'classification', model.n_neighbors = 5, model.weights = 'uniform', model.preprocess = 'None'
Accuracy: 0.9688

Decision tree model
Model parameters: model.task = 'classification', model.criterion = 'gini', model.max_depth = None
Accuracy: 0.8784


### Optimized Classification Models

The same wrapper interfaces can perform validation-based parameter selection for the MNIST classification task.

> **Runtime note:** optimized k-NN on the full MNIST dataset can be computationally expensive because nearest-neighbor prediction requires many high-dimensional distance calculations.


In [9]:
print('\nK-nn model')
model = knn()
model.fit(X_train,y_train,optimize=True)
print(f'Model parameters: {model.task = }, {model.n_neighbors = }, {model.weights = }, {model.preprocess = }')
pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test,pred):6.4f}')

print('\nDecision tree model')
model = decision_tree()
model.fit(X_train,y_train,optimize=True)
print(f'Model parameters: {model.task = }, {model.criterion = }, {model.max_depth = }')
pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test,pred):6.4f}')


K-nn model
Model parameters: model.task = 'classification', model.n_neighbors = 3, model.weights = 'distance', model.preprocess = 'None'
Accuracy: 0.9717

Decision tree model
Model parameters: model.task = 'classification', model.criterion = 'entropy', model.max_depth = 15
Accuracy: 0.8868


## 6. Saved Regression Results

| Model | Configuration | MAE | MSE |
|---|---|---:|---:|
| k-NN | 5 neighbors, uniform, no preprocessing | 0.8152 | 1.1420 |
| Decision tree | squared error, unrestricted depth | **0.4571** | **0.5235** |

## 7. Design Takeaways

- A shared wrapper can expose a consistent API across regression and classification.
- k-NN optimization naturally combines preprocessing selection with neighborhood and weighting choices.
- Decision-tree optimization can search criterion and maximum depth using task-appropriate validation metrics.
- The default California Housing comparison shows a substantially lower saved MSE for the unrestricted regression tree than for the unscaled five-neighbor regressor.